In [4]:
import pandas as pd
import numpy as np

In [5]:
url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
df = pd.read_csv(url, sep='\t', names=['user_id','item_id','rating','timestamp'])
df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [6]:
user_item_matrix = df.pivot(index='user_id', columns='item_id', values='rating')
user_item_matrix = user_item_matrix.fillna(0)
user_item_matrix.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
U, sigma, Vt = np.linalg.svd(user_item_matrix, full_matrices=False)
sigma = np.diag(sigma)

In [8]:
predicted_ratings = np.dot(np.dot(U, sigma), Vt)
pred_df = pd.DataFrame(predicted_ratings, columns=user_item_matrix.columns)

In [9]:
def recommend_movies(user_id, n=5):
    user_row = pred_df.iloc[user_id - 1]
    sorted_movies = user_row.sort_values(ascending=False)
    already_rated = user_item_matrix.iloc[user_id - 1]
    already_rated = already_rated[already_rated > 0].index
    recommendations = [movie for movie in sorted_movies.index if movie not in already_rated]
    return recommendations[:n]
print(recommend_movies(1))

[474, 434, 529, 513, 462]


In [10]:
from sklearn.metrics import mean_squared_error
# Only compare non-zero values
actual = user_item_matrix.values[user_item_matrix.values > 0]
predicted = predicted_ratings[user_item_matrix.values > 0]
rmse = np.sqrt(mean_squared_error(actual, predicted))
print("RMSE:", rmse)

RMSE: 7.291252382528203e-15
